# Notebook 13: registered statistics and visual evidence
Purpose: analyze the registered eight-seed initialization/final or six-seed final-only JEPA study on reused Food-101 development data. Inputs: registered plan, every declared fixed-epoch probe prediction and saved geometry/cost records. Outputs: seven PNGs, CSV evidence, geometry diagnostics and a self-contained HTML report under `<stage>/visual_report`. Expected runtime: 1–3 CPU minutes, no training/inference. Resume: import only this `ahr-export` cell and call `build_visual_report(root, artifact, plan, bundle, result)` after all endpoints complete. Register with `register_analysis13(root, artifact, plan)` before outcomes; source and plan hashes stay unchanged.
The three primary contrasts use Holm correction. Raw 95% and simultaneous Bonferroni 98.333% intervals are separate. Independent training sample size is the registered six or eight paired seeds; development images are reused, the earlier reserve is consumed, and official test is NOT_RUN.

In [ ]:
"""Registered eight-pair JEPA contrasts, reused-development visual evidence.
All executable logic belongs in this tagged notebook. No training or test-data reads.
"""
from pathlib import Path
import base64,csv,hashlib,html,itertools,json,math,os,time
import numpy as np
from scipy.stats import t as student_t,binomtest
import matplotlib
if "__file__" in globals():
    SOURCE13=Path(__file__).resolve()
else:
    SOURCE13=next((base/suffix for base in [Path.cwd(),*Path.cwd().parents]
        for suffix in ["experiments/13_product_geometry/visualization.ipynb","fmm_jeppa/experiments/13_product_geometry/visualization.ipynb"]
        if (base/suffix).exists()),None)
    if SOURCE13 is None:raise RuntimeError("Run within fmm_jeppa workspace")
os.environ["MPLCONFIGDIR"]=str(SOURCE13.parents[2]/".cache/matplotlib13")
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import ImageOps

VARIANTS=("vit","learned_kernel","normalized_kernel","product_kernel","sphere_kernel","multi_view_kernel")
PRIMARY_VARIANTS=("product_kernel","sphere_kernel","multi_view_kernel")
NAMES={"vit":"ViT","learned_kernel":"Learned kernel","normalized_kernel":"Normalized kernel","product_kernel":"Product kernel","sphere_kernel":"Sphere kernel","multi_view_kernel":"Multiple-view kernel"}
COLORS={"vit":"#2D6698","learned_kernel":"#8C9297","normalized_kernel":"#B8791B","product_kernel":"#A94F83","sphere_kernel":"#6C7835","multi_view_kernel":"#DC7553"}


def _sha13(path):
    digest=hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda:stream.read(1024*1024),b""):digest.update(block)
    return digest.hexdigest()


def _json13(path):return json.loads(Path(path).read_text())


def _canonical13(value):return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(",",":"),allow_nan=False).encode()).hexdigest()


def _write13(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temporary=path.with_name(path.name+".partial")
    with temporary.open("w") as stream:
        json.dump(value,stream,indent=2,sort_keys=True,allow_nan=False);stream.write("\n");stream.flush();os.fsync(stream.fileno())
    os.replace(temporary,path)


def _deadline13(deadline):
    if deadline is not None and time.time()>=float(deadline):raise TimeoutError("Visual deadline reached; report remains incomplete")


def _endpoints13(plan):
    final=int(plan["updates"]);declared=plan.get("probe_endpoints",["initial","final"])
    values=[0 if value=="initial" else final if value=="final" else int(value) for value in declared]
    if values not in [[final],[0,final]]:raise ValueError("Declared probe endpoints must be final only or initialization plus final")
    return values


def _pstr13(value):return "UNDEFINED" if value is None else f"{value:.4f}"
def _scorestr13(value):return "NOT EVALUATED" if value is None else f"{value:.3f}%"
def _gainstr13(value):return "NOT EVALUATED" if value is None else f"{value:+.3f}"
def _intervalstr13(low,high):return "UNDEFINED" if low is None or high is None else f"[{low:+.3f}, {high:+.3f}]"


def _final_only_quality13(folder,index,plan):
    seeds=list(map(int,plan["seeds"]));final=int(plan["updates"]);fig,axis=plt.subplots(figsize=(11,5));rows=[]
    for i,variant in enumerate(VARIANTS):
        values=[index[(variant,seed,final)]["row"]["top1_percent"] for seed in seeds]
        axis.scatter(np.full(len(seeds),i)+np.linspace(-.14,.14,len(seeds)),values,color=COLORS[variant],marker="D",s=38)
        axis.plot([i-.2,i+.2],[np.mean(values)]*2,color=COLORS[variant],linewidth=3)
        rows.append({"variant":variant,"initial_mean_top1_percent":None,"final_mean_top1_percent":float(np.mean(values)),"mean_learning_gain_pp":None,"ci95_gain_lower_pp":None,"ci95_gain_upper_pp":None})
    axis.set(xticks=range(6),xticklabels=[NAMES[v].replace(" ","\n",1) for v in VARIANTS],ylabel="Development top-1 accuracy (%)",title="Registered final models only; initialization was not evaluated")
    axis.grid(axis="y",color="#E4E8EA",alpha=.7)
    return _figure13(fig,folder,"quality_and_learning.png","Final encoder accuracy; no initialization inference",
        "The bounded final-only protocol evaluates every registered seed and encoder at fixed epoch20. Initialization quality and own-learning gain were not evaluated and remain unavailable. Reused development data; the same image IDs across all models.",rows),rows


def paired_stats13(deltas,seed_ids=None):
    values=np.asarray(deltas,dtype=np.float64)
    count=len(values) if values.ndim==1 else 0
    if count not in [6,8] or not np.isfinite(values).all():raise ValueError("Six or eight finite registered training-pair differences are required")
    if seed_ids is not None and (len(seed_ids)!=count or len(set(seed_ids))!=count):raise ValueError("Unique paired seed identities are required")
    df=count-1
    mean=float(values.mean());sd=float(values.std(ddof=1));se=sd/math.sqrt(count)
    if se==0:
        statistic=None;pvalue=1.0 if mean==0 else None;ci95=[mean,mean];cifamily=[mean,mean];degenerate=True
    else:
        statistic=mean/se;pvalue=float(2*student_t.sf(abs(statistic),df));degenerate=False
        radius95=float(student_t.ppf(.975,df)*se);radiusfamily=float(student_t.ppf(1-.05/(2*3),df)*se)
        ci95=[mean-radius95,mean+radius95];cifamily=[mean-radiusfamily,mean+radiusfamily]
    signs=np.asarray(list(itertools.product((-1,1),repeat=count)),dtype=np.int8)
    flipped=(signs*values[None,:]).mean(axis=1)
    pflip=float(np.mean(np.abs(flipped)>=abs(mean)-1e-12*max(1,abs(mean))))
    positive=int(np.sum(values>0));negative=int(np.sum(values<0));nonzero=positive+negative
    return {"n_training_seeds":count,"degrees_of_freedom":df,"mean_percentage_points":mean,"sd_percentage_points":sd,
        "standard_error_percentage_points":se,"approximate_t_ci95_percentage_points":ci95,
        "bonferroni_simultaneous_ci983333_percentage_points":cifamily,
        "simultaneous_interval_confidence_level":1-.05/3,"familywise_coverage_target":.95,"family_size":3,
        "t_statistic":statistic,"paired_t_two_sided_p_value":pvalue,"zero_variance_deltas":degenerate,
        "positive_seeds":positive,"negative_seeds":negative,"zero_seeds":count-nonzero,
        "exact_two_sided_signflip":{"p_value":pflip,"assignments":2**count,"means_percentage_points":flipped.tolist(),
            "assumption":"Independent paired differences symmetric under null, or pairwise architecture-label exchangeability."},
        "exact_two_sided_sign_test":{"p_value":float(binomtest(positive,nonzero,.5).pvalue) if nonzero else 1.,
            "positive_pairs":positive,"negative_pairs":negative,"tied_pairs_excluded":count-nonzero},
        "paired_seed_deltas":[{"seed":int(seed_ids[i]) if seed_ids is not None else i,"delta_percentage_points":float(value)} for i,value in enumerate(values)],
        "uncertainty":"Independent registered training pairs, conditional on the reused development image membership. Paired t inference assumes an appropriate paired-effect distribution; image count is not independent training replication."}


def holm_adjust13(pvalues):
    undefined=[value is None for value in pvalues]
    values=np.asarray([1. if value is None else value for value in pvalues],dtype=np.float64)
    if values.shape!=(3,) or not np.isfinite(values).all() or np.any(values<0) or np.any(values>1):raise ValueError("Exactly three valid primary p values required")
    order=np.argsort(values,kind="stable");adjusted=np.empty(3);previous=0.
    for rank,index in enumerate(order):
        previous=max(previous,(3-rank)*values[index]);adjusted[index]=min(1.,previous)
    return [None if undefined[index] else float(value) for index,value in enumerate(adjusted)]


def holm_primary13(chart_rows,seeds,updates):
    if len(seeds) not in [6,8] or len(set(seeds))!=len(seeds):raise ValueError("Prespecified six or eight independent training pairs required")
    lookup={}
    for row in chart_rows:
        key=(row["variant"],int(row["seed"]),int(row["updates"]))
        if key in lookup:raise ValueError("Duplicate chart row")
        lookup[key]=row
    comparisons={};keys=[]
    for variant in PRIMARY_VARIANTS:
        key=f"{variant}_minus_vit_u{updates}";keys.append(key)
        values=[float(lookup[(variant,int(seed),int(updates))]["top1_percent"])-float(lookup[("vit",int(seed),int(updates))]["top1_percent"]) for seed in seeds]
        comparisons[key]=paired_stats13(values,seeds)
    adjusted=holm_adjust13([comparisons[key]["paired_t_two_sided_p_value"] for key in keys])
    for key,pvalue in zip(keys,adjusted):
        stats=comparisons[key];stats["holm_adjusted_p_value"]=pvalue
        stats["superiority_supported"]=bool(stats["mean_percentage_points"]>0 and pvalue is not None and pvalue<.05 and not stats["zero_variance_deltas"])
        stats["conservative_simultaneous_interval_positive"]=bool(stats["bonferroni_simultaneous_ci983333_percentage_points"][0]>0 and not stats["zero_variance_deltas"])
    return {"registered":True,"family":keys,"alpha":.05,"multiplicity":"Holm","comparisons":comparisons,"n_pairs":len(seeds),
        "decision":"Positive mean and Holm-adjusted two-sided p<.05 for each of the three fixed primary contrasts. The Bonferroni interval criterion is displayed separately and may be more conservative.",
        "scope":"New paired training seeds, reused 2,020-image development subset. The previously consumed reserve is not fresh; official heldout test is NOT_RUN."}


def register_analysis13(root,artifact,plan):
    root,artifact=Path(root).resolve(),Path(artifact).resolve()
    if not artifact.is_relative_to(root):raise ValueError("Registration outside project")
    count=len(plan["seeds"])
    if count not in [6,8] or tuple(plan["variants"])!=VARIANTS or list(map(int,plan["seeds"]))!=list(range(20261501,20261501+count)):raise ValueError("Registered architectures/seeds differ")
    registration={"schema":"notebook13-registered-analysis/v1","analysis_notebook":str(SOURCE13),"analysis_notebook_sha256":_sha13(SOURCE13),
        "plan_hash":_canonical13(plan),"seeds":list(plan["seeds"]),"variants":list(VARIANTS),"updates":int(plan["updates"]),"probe_endpoints":plan.get("probe_endpoints",["initial","final"]),
        "primary_contrasts":[f"{variant}_minus_vit_u{plan['updates']}" for variant in PRIMARY_VARIANTS],
        "alpha":.05,"primary_test":"Registered paired training-seed two-sided t comparisons, Holm step-down family of three",
        "positive_superiority_rule":"Positive mean and Holm-adjusted two-sided p < .05; no extra Bonferroni interval condition",
        "intervals":{"raw":"95% paired t, per-comparison unadjusted","simultaneous":"98.333333% per-comparison Bonferroni t, >=95% simultaneous family coverage under paired-t assumptions"},
        "secondary":"All 2^n exact sign flips and sign tests; normalized-minus-old, geometry-minus-normalized, learning gains, quality and geometry visuals are exploratory",
        "selection":"All registered seeds and all six variants, declared probe endpoints and fixed final update; no quality-based model, probe-epoch, seed or class selection",
        "gallery":"First registered seed, lexicographically first development IDs from class labels 0,50,100; primary prediction comparisons illustrated only",
        "official_test":"NOT_RUN","development_status":"REUSED_PREVIOUSLY_REPORTED_2020_IMAGES","reserve_status":"PREVIOUSLY_CONSUMED_NOT_FRESH"}
    path=artifact/"analysis_registration.json"
    if path.exists():
        if _json13(path)!=registration:raise ValueError("Immutable analysis registration differs")
    else:
        if (artifact/"stage_evaluation.json").exists() or any((artifact/"evaluation").glob("*.json")):raise ValueError("Cannot newly preregister after quality evaluation outcomes exist")
        _write13(path,registration)
    return {"status":"REGISTERED","path":str(path),"sha256":_sha13(path),"source_sha256":registration["analysis_notebook_sha256"]}


def _prediction13(root,artifact,saved):
    path=Path(artifact)/"predictions"/(saved["model_id"]+".csv")
    if not path.exists():path=Path(saved["probe"]["summary"]["predictions_path"])
    if not path.resolve().is_relative_to(Path(root).resolve()):raise ValueError("Prediction path outside project")
    with path.open() as stream:rows=list(csv.DictReader(stream))
    ids=[row["sample_id"] for row in rows];labels=np.asarray([int(row["label"]) for row in rows]);predictions=np.asarray([int(row["top1_prediction"]) for row in rows])
    if len(ids)!=2020 or len(set(ids))!=2020 or set(labels.tolist())!=set(range(101)):raise ValueError("The reused development image membership is incomplete")
    score=100*float(np.mean(labels==predictions))
    summary=saved["probe"]["summary"]
    if int(summary["completed_epochs"])!=20 or summary["selection"]!="fixed_final_epoch":raise ValueError("Fixed epoch20 probe policy violated")
    if abs(score-100*float(summary["metrics"]["top1"]))>1e-4:raise ValueError("Prediction accuracy differs from summary")
    return ids,labels,predictions,score,{"path":str(path),"sha256":_sha13(path)}


def _evidence13(root,artifact,plan,result):
    if result.get("status")!="COMPLETE":raise ValueError("Final statistical report requires every declared endpoint evaluation")
    if tuple(plan["variants"])!=VARIANTS or len(plan["seeds"]) not in [6,8]:raise ValueError("Unexpected registered study design")
    final=int(plan["updates"]);endpoints=_endpoints13(plan);expected={(variant,int(seed),update) for variant in VARIANTS for seed in plan["seeds"] for update in endpoints}
    index={};rows=[];sources=[];common_ids=None;common_labels=None
    oldcharts={(row["variant"],int(row["seed"]),int(row["updates"])):row for row in result.get("chart_rows",[])}
    for saved in result["evaluations"].values():
        key=(saved["variant"],int(saved["seed"]),int(saved["updates"]))
        if key not in expected or key in index:raise ValueError("Unexpected or repeated endpoint evaluation")
        ids,labels,predictions,score,source=_prediction13(root,artifact,saved)
        if common_ids is None:common_ids,common_labels=ids,labels
        elif ids!=common_ids or not np.array_equal(labels,common_labels):raise ValueError("Development IDs and labels do not pair across encoders")
        row=dict(oldcharts.get(key,{}));row.update({"variant":key[0],"seed":key[1],"updates":key[2],"top1_percent":score})
        if key in oldcharts and abs(float(oldcharts[key]["top1_percent"])-score)>1e-4:raise ValueError("Recorded chart score does not reconcile with predictions")
        index[key]={"saved":saved,"predictions":predictions,"row":row};rows.append(row);sources.append(source)
    if set(index)!=expected:raise ValueError("All declared endpoint evaluations are required; missing endpoints are UNKNOWN")
    registered_ids=result.get("membership",{}).get("development_ids")
    if registered_ids is not None and list(registered_ids)!=common_ids:raise ValueError("Registered reused membership differs")
    for row in rows:
        initial=index.get((row["variant"],row["seed"],0))
        row["initial_top1_percent"]=initial["row"]["top1_percent"] if initial else None
        row["gain_percentage_points"]=row["top1_percent"]-row["initial_top1_percent"] if initial else None
    return index,rows,common_ids,common_labels,sources


def _figure13(fig,folder,name,title,note,evidence,hpad=2):
    if (Path(folder)/"fixture_marker.json").exists():fig.suptitle("SOFTWARE FIXTURE — NOT NOTEBOOK 13 RESULTS",fontsize=13,color="#B8791B")
    fig.tight_layout(h_pad=hpad,pad=1.5);path=Path(folder)/name
    fig.savefig(path,dpi=150,bbox_inches="tight",facecolor="white");plt.close(fig)
    return {"path":str(path),"sha256":_sha13(path),"title":title,"note":note,"evidence":evidence}


def _csv13(path,rows):
    if not rows:return
    keys=list(dict.fromkeys(key for row in rows for key in row))
    with Path(path).open("w",newline="") as stream:
        writer=csv.DictWriter(stream,fieldnames=keys);writer.writeheader();writer.writerows(rows)


def _quality13(folder,index,rows,plan):
    seeds=list(map(int,plan["seeds"]));final=int(plan["updates"]);positions=np.arange(6);jitter=np.linspace(-.15,.15,len(seeds))
    if 0 not in _endpoints13(plan):return _final_only_quality13(folder,index,plan)
    fig,axes=plt.subplots(1,2,figsize=(14,6));quality=[]
    for variant_index,variant in enumerate(VARIANTS):
        initial=[index[(variant,seed,0)]["row"]["top1_percent"] for seed in seeds]
        trained=[index[(variant,seed,final)]["row"]["top1_percent"] for seed in seeds]
        axes[0].scatter(positions[variant_index]-.16+jitter/3,initial,marker="o",facecolors="white",edgecolors=COLORS[variant],s=30)
        axes[0].scatter(positions[variant_index]+.16+jitter/3,trained,marker="D",color=COLORS[variant],s=30)
        axes[0].plot([positions[variant_index]-.16,positions[variant_index]+.16],[np.mean(initial),np.mean(trained)],color=COLORS[variant],linewidth=2)
        gain=np.asarray(trained)-initial;stats=paired_stats13(gain,seeds);mean=stats["mean_percentage_points"];low,high=stats["approximate_t_ci95_percentage_points"]
        axes[1].scatter(np.full(len(seeds),variant_index)+jitter,gain,color=COLORS[variant],s=22,alpha=.7)
        axes[1].errorbar(variant_index,mean,yerr=[[mean-low],[high-mean]],fmt="D",color=COLORS[variant],capsize=4,markersize=8,zorder=4)
        quality.append({"variant":variant,"initial_mean_top1_percent":float(np.mean(initial)),"final_mean_top1_percent":float(np.mean(trained)),"mean_learning_gain_pp":mean,"ci95_gain_lower_pp":low,"ci95_gain_upper_pp":high})
    labels=[NAMES[v].replace(" ","\n",1) for v in VARIANTS]
    axes[0].set(xticks=positions,xticklabels=labels,ylabel="Development top-1 accuracy (%)",title="Initialization (open) and final (filled)")
    axes[1].set(xticks=positions,xticklabels=labels,ylabel="Final − own initialization (pp)",title="Exploratory learning gains; registered paired seeds")
    axes[1].axhline(0,color="#263238",linewidth=1)
    for axis in axes:axis.grid(axis="y",color="#E4E8EA",alpha=.7)
    return _figure13(fig,folder,"quality_and_learning.png","All encoder endpoints and learning gains",
        "All declared fixed epoch 20 probes, every registered seed. Open circles show initialization, filled diamonds final training; lines connect architecture means. Gain intervals are unadjusted 95% paired t intervals, exploratory. Development membership is reused.",quality),quality


def _primary_figure13(folder,primary):
    fig,axes=plt.subplots(1,2,figsize=(14,5));rows=[];count=primary["n_pairs"]
    for i,variant in enumerate(PRIMARY_VARIANTS):
        key=primary["family"][i];stats=primary["comparisons"][key];mean=stats["mean_percentage_points"]
        raw=stats["approximate_t_ci95_percentage_points"];sim=stats["bonferroni_simultaneous_ci983333_percentage_points"]
        if not stats["zero_variance_deltas"]:
            axes[0].errorbar(mean,i,xerr=[[mean-sim[0]],[sim[1]-mean]],fmt="none",color=COLORS[variant],capsize=5,linewidth=1.5)
            axes[0].errorbar(mean,i,xerr=[[mean-raw[0]],[raw[1]-mean]],fmt="D",color=COLORS[variant],capsize=0,linewidth=4,markersize=7)
        else:axes[0].scatter([mean],[i],marker="x",color=COLORS[variant],s=60)
        values=[item["delta_percentage_points"] for item in stats["paired_seed_deltas"]]
        axes[1].scatter(values,np.full(count,i)+np.linspace(-.1,.1,count),color=COLORS[variant],s=30)
        label=f"{NAMES[variant]}\nHolm p={_pstr13(stats['holm_adjusted_p_value'])}"
        rows.append({"variant":variant,"mean_difference_pp":mean,"raw_ci95_low_pp":raw[0] if not stats["zero_variance_deltas"] else None,"raw_ci95_high_pp":raw[1] if not stats["zero_variance_deltas"] else None,
            "bonferroni_ci983333_low_pp":sim[0] if not stats["zero_variance_deltas"] else None,"bonferroni_ci983333_high_pp":sim[1] if not stats["zero_variance_deltas"] else None,"variance_degenerate":stats["zero_variance_deltas"],"raw_two_sided_p":stats["paired_t_two_sided_p_value"],
            "holm_adjusted_p":stats["holm_adjusted_p_value"],"holm_positive_superiority_supported":stats["superiority_supported"],
            "conservative_interval_positive":stats["conservative_simultaneous_interval_positive"],"signflip_p":stats["exact_two_sided_signflip"]["p_value"]})
    ylabels=[f"{NAMES[v]}\nHolm p={_pstr13(primary['comparisons'][primary['family'][i]]['holm_adjusted_p_value'])}" for i,v in enumerate(PRIMARY_VARIANTS)]
    for axis in axes:
        axis.axvline(0,color="#263238",linewidth=1);axis.set(yticks=range(3),yticklabels=ylabels,xlabel="Final encoder − ViT top-1 (percentage points)")
    axes[0].set_title("Primary: thick 95% raw, thin 98.333% simultaneous")
    axes[1].set_title("All registered independent paired training-seed effects")
    return _figure13(fig,folder,"registered_primary_family.png","Registered primary contrasts and family adjustment",
        "Three fixed primary contrasts use two-sided paired t tests, df=n−1, with Holm correction at family α=.05; positive mean and adjusted p<.05 decide support. Thick intervals are raw 95%; thin intervals are Bonferroni 98.333% per comparison (at least 95% simultaneous family coverage), a separately conservative criterion. n registered training pairs; image count is not independent training replication. Crosses mark degenerate seed variance, for which t intervals and positive superiority are unsupported.",primary),rows


def _cost13(folder,rows,final):
    fig,axes=plt.subplots(1,3,figsize=(15,5));cost=[]
    fields=[("median_training_step_seconds","Full JEPA median update time (seconds)"),("peak_training_GiB","Peak training allocated CUDA memory (GiB)"),("trainable_active_parameters","Active trainable parameters (millions)")]
    for i,variant in enumerate(VARIANTS):
        selected=[row for row in rows if row["variant"]==variant and row["updates"]==final];record={"variant":variant}
        for axis,(key,label) in zip(axes,fields):
            values=[float(row[key]) for row in selected if row.get(key) is not None]
            if not values:record[key]=None;axis.text(i,.02,"UNKNOWN",rotation=90,ha="center",va="bottom",transform=axis.get_xaxis_transform(),fontsize=9);continue
            values=np.asarray(values)/(1e6 if key=="trainable_active_parameters" else 1)
            axis.scatter(np.full(len(values),i)+np.linspace(-.13,.13,len(values)),values,color=COLORS[variant],s=20,alpha=.7)
            axis.plot([i-.2,i+.2],[values.mean(),values.mean()],color=COLORS[variant],linewidth=3)
            record[key]=float(values.mean());record[key+"_measured_seeds"]=len(values)
        cost.append(record)
    for axis,(_,label) in zip(axes,fields):
        axis.set(xticks=range(6),xticklabels=["ViT","Learned","Norm.","Product","Sphere","3-view"],ylabel=label,ylim=(0,max(1e-12,axis.get_ylim()[1])*1.1));axis.grid(axis="y",color="#E4E8EA",alpha=.7)
    return _figure13(fig,folder,"full_jepa_training_cost.png","Full JEPA training cost and capacity",
        "Final-model costs come from authoritative training histories, covering the online encoder, EMA target, predictor and masked latent objective; this is not an isolated kernel benchmark. Dots show all registered seeds; lines show means. Norm. denotes Normalized kernel and 3-view denotes Multiple-view kernel. CUDA memory excludes feature/probe evaluation. Data preparation and recorded optimizer work contribute to update timing. UNKNOWN remains visible when cost evidence is unavailable. Dense kernel sums are used; no FMM speedup is established.",cost),cost


def _geometry13(folder,index,plan):
    seed=int(plan["seeds"][0]);final=int(plan["updates"]);records={};flat=[]
    for variant in VARIANTS[1:]:
        saved=index[(variant,seed,final)]["saved"];geometry=saved.get("kernel_geometry_diagnostics",{})
        layers=geometry.get("layers",[])
        records[variant]={"status":geometry.get("status","UNKNOWN"),"scope":geometry.get("scope","UNKNOWN"),"layers":layers}
        for layer in layers:flat.append({"variant":variant,**layer})
    fig,axes=plt.subplots(1,2,figsize=(13,5));labels=[NAMES[v] for v in VARIANTS[1:]]
    for axis,key,title in zip(axes,["bandwidths","content_bandwidths"],["Spatial bandwidths","Content bandwidths"]):
        for i,variant in enumerate(VARIANTS[1:]):
            values=[]
            for layer in records[variant]["layers"]:
                if layer.get(key) is not None:values.extend(np.asarray(layer[key],dtype=float).ravel().tolist())
            if values:
                if min(values)<=0 or not np.isfinite(values).all():raise ValueError("Geometry bandwidths must remain positive and finite")
                axis.scatter(values,np.full(len(values),i)+np.linspace(-.12,.12,len(values)),color=COLORS[variant],s=15,alpha=.55)
                axis.plot([np.median(values)]*2,[i-.22,i+.22],color=COLORS[variant],linewidth=3)
            else:axis.text(.5,i,"UNKNOWN",ha="center",transform=axis.get_yaxis_transform())
        axis.set(yscale="linear",xscale="log",yticks=range(5),yticklabels=labels,xlabel="Dimensionless kernel length scale",title=title)
    bandwidth=_figure13(fig,folder,"geometry_bandwidths.png","Learned bandwidth distributions",
        f"Fixed first registered seed {seed}, final model; every block, head and mixture component retained. Each dot is one recorded length scale; a vertical mark shows the median. Spatial and content axes use log scales. Euclidean content and sphere chord coordinates have different geometries; bandwidth comparisons are descriptive.",records)
    fig,axes=plt.subplots(1,3,figsize=(15,5));mix_rows=[]
    for i,variant in enumerate(VARIANTS[1:]):
        layers=records[variant]["layers"];mixtures=[np.asarray(layer["mixture_weights"],float).reshape(-1,3) for layer in layers if layer.get("mixture_weights") is not None]
        if mixtures:
            joined=np.concatenate(mixtures)
            if not np.allclose(joined.sum(axis=1),1,atol=1e-5) or np.any(joined<0):raise ValueError("Kernel mixture mass invalid")
            mean=joined.mean(axis=0);mix_rows.append({"variant":variant,"mixture_component_means":mean.tolist()})
            for component,value in enumerate(mean):axes[0].scatter(component,float(value),color=COLORS[variant],marker=["o","s","^","D","P"][i],s=50,label=NAMES[variant] if component==0 else None)
        for key,style in [("effective_neighbors_exp_entropy_mean","-"),("effective_neighbors_participation_mean","--")]:
            points=[(int(layer.get("block",j)),float(layer[key])) for j,layer in enumerate(layers) if layer.get(key) is not None]
            if points:axes[1].plot([p[0] for p in points],[p[1] for p in points],style,color=COLORS[variant],marker="o",label=NAMES[variant] if style=="-" else None)
    axes[0].set(xticks=range(3),xticklabels=["Small","Medium","Large"],ylim=(0,1.02),ylabel="Mean normalized mixture weight",title="Kernel mixture mass")
    axes[0].legend(frameon=False,fontsize=8)
    axes[1].set(xticks=range(4),xlabel="Encoder block",ylabel="Effective number of source tokens",title="Neighbors: entropy (solid), participation (dashed)")
    axes[1].legend(frameon=False,fontsize=8)
    layers=records["multi_view_kernel"]["layers"]
    for key,style in [("source_membership_mean","-"),("receiver_membership_mean","--")]:
        gate=[np.asarray(layer[key],float) for layer in layers if layer.get(key) is not None]
        if gate:
            gate=np.asarray(gate)
            if gate.shape[1]!=3 or not np.allclose(gate.sum(axis=1),1,atol=1e-5):raise ValueError("View mixing mass invalid")
            axes[2].plot(range(3),gate.mean(axis=0),style,color=COLORS["multi_view_kernel"],marker="o",label=key.split("_")[0].capitalize())
    axes[2].set(xticks=range(3),xticklabels=["View1","View2","View3"],ylim=(0,1.02),ylabel="Mean three-view membership",title="Multiple-view source and receiver gates")
    if axes[2].lines:axes[2].legend(frameon=False)
    else:axes[2].text(.5,.5,"UNKNOWN",transform=axes[2].transAxes,ha="center")
    mixture=_figure13(fig,folder,"geometry_mass_and_neighbors.png","Mixture mass, neighbors and view membership",
        f"Descriptive first seed {seed}, final geometry on fixed diagnostic images. Mixture plots average block/head values, retaining all layers in evidence. Effective neighbor counts use normalized kernel row weights. Three-view gates show shared source memberships and receiver mixing; these do not establish a discovered data manifold.",records)
    fig,axes=plt.subplots(1,2,figsize=(12,4));norm_records=[]
    for variant in ["sphere_kernel","multi_view_kernel"]:
        layers=records[variant]["layers"]
        for key,style in [("query_unit_norm_max_error","-"),("source_unit_norm_max_error","--")]:
            points=[(int(layer.get("block",j)),float(layer[key])) for j,layer in enumerate(layers) if layer.get(key) is not None]
            if points:
                axes[0].plot([p[0] for p in points],[max(p[1],1e-12) for p in points],style,color=COLORS[variant],marker="o",label=NAMES[variant]+" "+key.split("_")[0])
                norm_records.append({"variant":variant,"key":key,"values":points})
        points=[(int(layer.get("block",j)),float(layer["zero_fallback_count"])) for j,layer in enumerate(layers) if layer.get("zero_fallback_count") is not None]
        if points:axes[1].plot([p[0] for p in points],[p[1] for p in points],color=COLORS[variant],marker="o",label=NAMES[variant])
    axes[0].set(xticks=range(4),yscale="log",xlabel="Encoder block",ylabel="Maximum |coordinate norm −1|",title="Sphere coordinate normalization")
    axes[1].set(xticks=range(4),xlabel="Encoder block",ylabel="Recorded zero-vector fallback count",title="Explicit zero-coordinate fallback",ylim=(0,max(1,axes[1].get_ylim()[1])))
    from matplotlib.ticker import MaxNLocator
    axes[1].yaxis.set_major_locator(MaxNLocator(integer=True))
    for axis in axes:
        if axis.lines:axis.legend(frameon=False,fontsize=8)
        else:axis.text(.5,.5,"UNKNOWN",transform=axis.transAxes,ha="center")
    norms=_figure13(fig,folder,"sphere_normalization_checks.png","Sphere coordinate numerical checks",
        "Query/source normalization errors are numerical checks of the sphere constraint. Multiple-view coordinate diagnostics cover the first reported view; memberships cover all three views. Exact zeros display at 1e−12 on the logarithmic axis only; recorded values remain in evidence. Unavailable fallback counts remain UNKNOWN.",records)
    return [bandwidth,mixture,norms],records


def _gallery13(root,folder,bundle,index,plan,ids,labels):
    dataset=bundle["dataset"];seed=int(plan["seeds"][0]);final=int(plan["updates"]);selected=[]
    for label in [0,50,100]:
        choices=sorted([i for i,value in enumerate(labels) if int(value)==label],key=lambda i:ids[i])
        selected.append(choices[0])
    fig,axes=plt.subplots(1,3,figsize=(13,6));records=[]
    for axis,position in zip(axes,selected):
        source=dataset.get(ids[position]);image=ImageOps.fit(source["image"],(350,260));axis.imshow(image);axis.set_axis_off()
        true=dataset.class_names[int(labels[position])].replace("_"," ");texts=["True: "+true]
        predictions={}
        for variant in ["vit",*PRIMARY_VARIANTS]:
            predicted=int(index[(variant,seed,final)]["predictions"][position]);predictions[variant]=predicted
            texts.append(NAMES[variant]+": "+dataset.class_names[predicted].replace("_"," "))
        axis.set_title("\n".join(texts),fontsize=9)
        path=dataset.path_for(ids[position]);records.append({"sample_id":ids[position],"label":int(labels[position]),"predictions":predictions,"path":str(path),"sha256":_sha13(path)})
    return _figure13(fig,folder,"illustrative_images.png","Fixed image examples",
        f"First registered seed {seed}; lexicographically first development ID in fixed classes 0, 50, 100. ViT and the three primary variants use the same images. Images are center-cropped for display only. Examples were fixed without reference to correctness and cannot determine statistical superiority.",records)


def _compare_primary13(derived,recorded):
    if not recorded:return
    comparisons=recorded.get("comparisons",{})
    for key,stats in derived["comparisons"].items():
        if key not in comparisons:raise ValueError("Workflow omitted registeredprimarycontrast")
        other=comparisons[key]
        for field in ["mean_percentage_points","paired_t_two_sided_p_value","holm_adjusted_p_value"]:
            if field in other:
                if stats[field] is None or other[field] is None:
                    if stats[field] is not other[field]:raise ValueError("Independent variance-degenerate statistics differ from workflow")
                elif abs(float(stats[field])-float(other[field]))>1e-4:raise ValueError("Independent visual statistics differ from workflow primary")
        if "superiority_supported" in other and bool(other["superiority_supported"])!=stats["superiority_supported"]:raise ValueError("Workflow andvisualprimary decisions differ")


def build_visual_report(root,artifact,plan,bundle,result,deadline_unix=None):
    root,artifact=Path(root).resolve(),Path(artifact).resolve();_deadline13(deadline_unix)
    if not artifact.is_relative_to(root):raise ValueError("Visual report outsideproject")
    fixture=bool(plan.get("visualization_fixture_source"))
    if not fixture:
        registration=_json13(artifact/"analysis_registration.json")
        if registration["analysis_notebook_sha256"]!=_sha13(SOURCE13) or registration["plan_hash"]!=_canonical13(plan):raise ValueError("Analysis orplan differs frompreregistration")
    index,rows,ids,labels,sources=_evidence13(root,artifact,plan,result)
    primary=holm_primary13(rows,list(map(int,plan["seeds"])),int(plan["updates"]));_compare_primary13(primary,result.get("primary_analysis"))
    folder=artifact/"visual_report";folder.mkdir(parents=True,exist_ok=True)
    if fixture:_write13(folder/"fixture_marker.json",{"fixture_source":plan["visualization_fixture_source"]})
    elif (folder/"fixture_marker.json").exists():raise ValueError("Actual output directory contains an old fixture marker")
    plt.rcParams.update({"font.family":"DejaVu Sans","font.size":10,"axes.spines.top":False,"axes.spines.right":False})
    figures=[];quality,qualityrows=_quality13(folder,index,rows,plan);figures.append(quality);_deadline13(deadline_unix)
    primfig,primaryrows=_primary_figure13(folder,primary);figures.append(primfig)
    costs,costrows=_cost13(folder,rows,int(plan["updates"]));figures.append(costs)
    geometry,geometryrows=_geometry13(folder,index,plan);figures.extend(geometry);_deadline13(deadline_unix)
    figures.append(_gallery13(root,folder,bundle,index,plan,ids,labels))
    if fixture:
        for figure in figures:figure["note"]="SOFTWARE TEST FIXTURE, NOT NOTEBOOK13 MODEL RESULTS. "+figure["note"]
    _csv13(folder/"endpoint_scores.csv",rows);_csv13(folder/"quality_summary.csv",qualityrows);_csv13(folder/"primary_family.csv",primaryrows);_csv13(folder/"training_cost.csv",costrows)
    _write13(folder/"geometry_evidence.json",geometryrows)
    stats={"schema":"notebook13-visual-analysis/v1","status":"COMPLETE","primary_analysis":primary,"scores":qualityrows,
        "n_pairs":len(plan["seeds"]),"probe_endpoints":plan.get("probe_endpoints",["initial","final"]),"primary_development_status":"REUSED_PREVIOUSLY_REPORTED_2020_IMAGES","reserve_status":"PREVIOUSLY_CONSUMED_NOT_FRESH","official_test":"NOT_RUN",
        "source_predictions":sources,"fixture_source":plan.get("visualization_fixture_source"),"new_training_in_this_report":"NOT_RUN",
        "scope":"Full custom JEPA training replaces the encoder while preserving EMA targets, predictor, masking and objective. Dense kernels are summed exactly; FMM acceleration is not demonstrated.",
        "limitations":["Only six or eight registered independent training pairs; paired t distribution assumptions matter.","Architecture ideas were developed using earlier development results; the reused subset is not independent heldout confirmation.","Holm controls the three prespecified primary comparisons; secondary charts and mechanism contrasts are exploratory.","Raw 95% intervals and Bonferroni 98.333% simultaneous intervals are separate; the Holm decision does not require the conservative interval to exclude zero.","Nonsignificance does not prove equivalence. Geometry statistics do not prove that coordinates form charts of a data manifold."]}
    _write13(folder/"analysis_results.json",stats);_write13(folder/"figure_evidence.json",{"status":"COMPLETE","fixture_source":stats["fixture_source"],"figures":figures})
    title="Product and sphere geometry: JEPA development comparison"
    prefix="SOFTWARE VALIDATION FIXTURE — NOT NEW MODEL RESULTS. " if fixture else ""
    parts=["<!doctype html><html lang='en'><meta charset='utf-8'><meta name='viewport' content='width=device-width,initial-scale=1'>",
        "<title>"+html.escape(prefix+title)+"</title><style>body{font-family:system-ui,sans-serif;max-width:1160px;margin:auto;padding:24px;color:#263238;line-height:1.5}img{width:100%;height:auto}figure{margin:32px 0}figcaption{font-size:.94rem;color:#495861}table{width:100%;border-collapse:collapse}th,td{padding:8px;border-bottom:1px solid #dce2e5;text-align:left}code{overflow-wrap:anywhere}</style>",
        "<h1>"+html.escape(prefix+title)+"</h1>",
        f"<p>{len(plan['seeds'])} paired seeds, six encoder architectures, fixed final update {plan['updates']}, {len(rows)} fixed epoch 20 linear probes. Every model uses the same 2,020 previously reported development images. The 8,074-image reserve was already consumed; the official test remains untouched.</p>",
        "<p>Primary family: Product, Sphere and Multiple-view encoders minus ViT. A positive paired mean and Holm-adjusted two-sided p&lt;.05 support improvement within this development protocol. Raw 95% paired t intervals and separate Bonferroni 98.333% intervals are displayed; the latter provides a conservative family interval criterion.</p>",
        "<table><tr><th>Primary contrast</th><th>Mean(pp)</th><th>Raw95%CI</th><th>Holm adjusted p</th><th>Positive Holm support</th></tr>"]
    for row in primaryrows:
        parts.append(f"<tr><td>{html.escape(NAMES[row['variant']])}−ViT</td><td>{row['mean_difference_pp']:+.3f}</td><td>{_intervalstr13(row['raw_ci95_low_pp'],row['raw_ci95_high_pp'])}</td><td>{_pstr13(row['holm_adjusted_p'])}</td><td>{row['holm_positive_superiority_supported']}</td></tr>")
    parts.append("</table><h2>All architecture endpoints</h2><table><tr><th>Encoder</th><th>Initialization mean</th><th>Final mean</th><th>Mean gain (pp)</th></tr>")
    for row in qualityrows:parts.append(f"<tr><td>{html.escape(NAMES[row['variant']])}</td><td>{_scorestr13(row['initial_mean_top1_percent'])}</td><td>{row['final_mean_top1_percent']:.3f}%</td><td>{_gainstr13(row['mean_learning_gain_pp'])}</td></tr>")
    parts.append("</table>")
    for figure in figures:
        encoded=base64.b64encode(Path(figure["path"]).read_bytes()).decode()
        parts.append("<figure><h2>"+html.escape(figure["title"])+"</h2><img alt='"+html.escape(figure["title"],quote=True)+"' src='data:image/png;base64,"+encoded+"'><figcaption>"+html.escape(figure["note"])+"</figcaption></figure>")
    parts.append("<h2>Scope and uncertainty</h2><ul>"+"".join("<li>"+html.escape(item)+"</li>" for item in stats["limitations"])+"</ul><p>CSV scores, primary contrasts, costs and geometry evidence are saved beside this report. Official test predictions and quality-based selection are absent.</p></html>")
    path=folder/"report.html";temporary=path.with_name(path.name+".partial");temporary.write_text("\n".join(parts));os.replace(temporary,path)
    _deadline13(deadline_unix)
    manifest={"status":"PASS","html":str(path),"html_sha256":_sha13(path),"fixture_source":stats["fixture_source"],"figures":[{key:value for key,value in figure.items() if key!="evidence"} for figure in figures],
        "analysis_results":str(folder/"analysis_results.json"),"analysis_results_sha256":_sha13(folder/"analysis_results.json"),"source_sha256":_sha13(SOURCE13),"browser_layout_validation":"NOT_RUN"}
    _write13(artifact/"visual_report_manifest.json",manifest)
    return {"status":"PASS","html":str(path),"report":str(path),"figures":manifest["figures"],"manifest":str(artifact/"visual_report_manifest.json"),"primary_analysis":primary,"fixture_source":stats["fixture_source"]}


def load_review13(root,artifact):
    artifact=Path(artifact);status=_json13(artifact.parent.parent/"study_status.json") if (artifact.parent.parent/"study_status.json").exists() else {}
    manifest=artifact/"visual_report_manifest.json"
    if not manifest.exists():return {"status":status.get("status","NOT_COMPLETE"),"figures":[],"html":None,"scope":"Saved progress only; no statistical conclusions before all declared completed endpoints"}
    saved=_json13(manifest)
    if _sha13(saved["html"])!=saved["html_sha256"]:raise ValueError("Savedreporthash differs")
    for figure in saved["figures"]:
        if _sha13(figure["path"])!=figure["sha256"]:raise ValueError("Savedfigurehash differs")
    return saved
